# Notebook 05c — Shadow Model Threat Model Comparison

How much does the shadow model initialisation matter?

All three variants use `split_mode='member_aware'` — the same protocol as NB05b.
The subject and pair lists come from NB01 outputs:
- `logs/subject_split.json` — member/non-member split (single source of truth)
- `logs/d5_attribution.npz` — pair-to-subject mapping, built by NB01's fingerprinting which handles the D5 label convention (raw 1=diff, 2=same) and the D1→D2 ID remapping for 5 non-member subjects

The only variable between the three variants is what the shadow LSTMs are initialised from:

| Variant | Init | Who provides the weights? |
|---|---|---|
| Grey-box | Real target model | Attacker obtained the authentication model checkpoint |
| Black-box warm | Attacker-trained surrogate | Attacker ran their own full training from scratch |
| Black-box cold | Random Xavier | Attacker knows only the architecture |

Grey-box results are loaded directly from the NB05b checkpoint — no retraining.
The gap grey − warm isolates the value of the exact target weights vs a self-trained surrogate.
The gap warm − cold isolates the value of a converged starting point vs random init.

Checkpoint files:
- `logs/05b_lira_target_shadows.json` — grey-box (produced by NB05b, member-aware)
- `logs/05c_lira_warm_shadows.json`   — black-box warm (surrogate init, member-aware)
- `logs/05c_lira_cold_shadows.json`   — black-box cold (random init, member-aware)

In [7]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
sys.path.insert(0, '..')

import json, logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_curve, auc as sk_auc

from src.data.auth_dataset    import load_auth_dataset, normalize_auth
from src.models.gait_cnn     import GaitCNN
from src.models.auth_model   import AuthModel
from src.attacks.lira_shadow import ShadowLSTM, train_shadow_models, lira_eval, tpr_at_fpr

DATA_ROOT  = Path('../data')
LOG_DIR    = Path('../logs')
CKPT_DIR   = Path('../checkpoints')
RESULT_DIR = Path('../results')
BATCH      = 512
DEVICE     = torch.device('cpu')

K_LIRA           = 32  # <<< RUNNER INJECTS THIS
LIRA_EPOCHS      = 3  # <<< RUNNER INJECTS THIS
SURROGATE_EPOCHS = 5  # <<< RUNNER INJECTS THIS
DROPOUT          = 0.3   # must match NB03 DROPOUT

# Grey-box: load from NB05b (no retraining)
# Black-box variants trained here

log = logging.getLogger('nb05c')
log.setLevel(logging.DEBUG)
log.handlers.clear()
fh = logging.FileHandler(LOG_DIR / '05c_shadow_comparison.log', mode='w')
fh.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter('%(message)s'))
log.addHandler(fh); log.addHandler(sh)
log.info('=== Notebook 05c — Shadow Model Threat Model Comparison ===')
# ── dataset config ──
import sys as _sys; _sys.path.insert(0, '..')
from src.utils.config_loader import get_dataset, dataset_dirs, dataset_files
from src.data.pairs_loader   import load_auth_pairs, load_attribution, build_pair_subjects_train, build_pair_subjects_test

DATASET = 'whuGAIT'  # <<< RUNNER INJECTS THIS
# Per-dataset subdirectories — isolates all artifacts by dataset
_ds_dirs   = dataset_dirs(Path('../logs'), Path('../checkpoints'), artifacts_base=Path('../artifacts'), results_base=Path('../results'), dataset=DATASET)
LOG_DIR    = _ds_dirs['logs']
ARTIFACT_DIR = _ds_dirs['artifacts']
CKPT_DIR   = _ds_dirs['checkpoints']
RESULT_DIR = _ds_dirs['results']
CKPT_GREY_SRC = ARTIFACT_DIR / f'05b_lira_target_shadows_{DATASET}.json'
CKPT_WARM     = ARTIFACT_DIR / f'05c_lira_warm_shadows_{DATASET}.json'
CKPT_COLD     = ARTIFACT_DIR / f'05c_lira_cold_shadows_{DATASET}.json'
PATHS   = dataset_files(LOG_DIR, CKPT_DIR, artifacts_dir=ARTIFACT_DIR)

with open(PATHS['split_json']) as _f:
    _split = json.load(_f)
member_ids    = _split['train_ids']
nonmember_ids = _split['held_out_ids']
N_CLASSES     = len(member_ids)
member_id_set    = set(member_ids)
nonmember_id_set = set(nonmember_ids)
log.info(f'Dataset: {DATASET}  |  N_CLASSES: {N_CLASSES}')

## Section 1 — Load Common State

In [ ]:
d = np.load(ARTIFACT_DIR / '05a_target_deltas.npz')
member_list    = d['member_ids'].tolist()
nonmember_list = d['nonmember_ids'].tolist()
all_mf_targets = {**dict(zip(d['member_ids'].tolist(), d['member_mf'].tolist())),
                  **dict(zip(d['nonmember_ids'].tolist(), d['nonmember_mf'].tolist()))}

with open(PATHS['split_json']) as f:
    subject_split = json.load(f)
member_id_set    = set(subject_split['train_ids'])
nonmember_id_set = set(subject_split['held_out_ids'])

norm_data = np.load(PATHS['norm_stats'])
norm_mean2, norm_std2 = norm_data['mean'], norm_data['std']

# Read encoder n_classes from metadata (cross-dataset: source may differ from N_CLASSES)
_cnn_n = json.load(open(PATHS['cnn_meta']))['n_classes'] if PATHS['cnn_meta'].exists() else N_CLASSES
cnn_base = GaitCNN(n_classes=_cnn_n)
cnn_base.load_state_dict(torch.load(PATHS['cnn_ckpt'], map_location='cpu'))
cnn_base.eval()

target_auth = AuthModel(cnn_base, dropout=DROPOUT).to(DEVICE)
target_auth.load_state_dict(torch.load(PATHS['auth_ckpt'], map_location='cpu'))
target_auth.eval()

X1_tr, X2_tr, y_tr = load_auth_pairs('train', DATA_ROOT, ARTIFACT_DIR)
X1_n, X2_n, _      = normalize_auth(X1_tr, X2_tr, (norm_mean2, norm_std2))
X1_te, X2_te, y_te = load_auth_pairs('test',  DATA_ROOT, ARTIFACT_DIR)
X1_te_n, X2_te_n, _ = normalize_auth(X1_te, X2_te, (norm_mean2, norm_std2))

subj_win1_tr, subj_win2_tr = load_attribution('train', ARTIFACT_DIR)
subj_win1_te, subj_win2_te = load_attribution('test',  ARTIFACT_DIR)
pair_subjects    = build_pair_subjects_train(y_tr, subj_win1_tr, subj_win2_tr, member_id_set, logs_dir=LOG_DIR)
te_pair_subjects = build_pair_subjects_test( y_te, subj_win1_te, subj_win2_te, nonmember_id_set, logs_dir=LOG_DIR)

print('Precomputing CNN features (~30s)...')
feats_list = []
with torch.no_grad():
    for s in range(0, len(X1_n), BATCH):
        e = min(s + BATCH, len(X1_n))
        feats_list.append(torch.cat([
            cnn_base.get_feature_maps(torch.from_numpy(X1_n[s:e]).float()),
            cnn_base.get_feature_maps(torch.from_numpy(X2_n[s:e]).float()),
        ], dim=1).half().cpu())
all_feats = torch.cat(feats_list, dim=0)

te_feats_list = []
with torch.no_grad():
    for s in range(0, len(X1_te_n), BATCH):
        e = min(s + BATCH, len(X1_te_n))
        te_feats_list.append(torch.cat([
            cnn_base.get_feature_maps(torch.from_numpy(X1_te_n[s:e]).float()),
            cnn_base.get_feature_maps(torch.from_numpy(X2_te_n[s:e]).float()),
        ], dim=1).half().cpu())
te_feats = torch.cat(te_feats_list, dim=0)
print(f'Done. Train {tuple(all_feats.shape)}  Test {tuple(te_feats.shape)}')

## Section 2 — Train Warm Surrogate

The black-box warm attacker cannot access the target model, but they do have access to gait pair data (e.g., collected independently). They run one full training of a ShadowLSTM on all available pairs to obtain a converged set of weights.

This surrogate model is then used as the shared starting point for all K warm shadows, making each shadow's fine-tuning faster and more stable than starting from random. The surrogate is NOT the target model — it was trained by the attacker, not the victim.

Training on all pairs for `SURROGATE_EPOCHS` epochs takes roughly the same time as one shadow training run.

In [10]:
torch.manual_seed(0)  # fix surrogate weights across kernel restarts
surrogate = ShadowLSTM(dropout=DROPOUT).to(DEVICE)
surrogate_opt = Adam(surrogate.parameters(), lr=0.0025, weight_decay=0.0015)
loss_fn       = nn.CrossEntropyLoss()

loader_full = DataLoader(
    TensorDataset(all_feats.float(), torch.from_numpy(y_tr).long()),
    batch_size=BATCH, shuffle=True, num_workers=0,
)

log.info(f'Training surrogate ({SURROGATE_EPOCHS} epochs on all pairs)...')
for epoch in range(SURROGATE_EPOCHS):
    surrogate.train()
    total_loss = 0.0
    for fb, yb in loader_full:
        fb, yb = fb.to(DEVICE), yb.to(DEVICE)
        surrogate_opt.zero_grad()
        l = loss_fn(surrogate(fb), yb)
        l.backward()
        surrogate_opt.step()
        total_loss += l.item()
    log.info(f'  epoch {epoch+1}/{SURROGATE_EPOCHS}  loss={total_loss/len(loader_full):.4f}')

surrogate_state = {
    'lstm': {k: v.clone() for k, v in surrogate.lstm.state_dict().items()},
    'fc':   {k: v.clone() for k, v in surrogate.fc.state_dict().items()},
}
del surrogate, surrogate_opt, loader_full, loss_fn
print('Surrogate trained. Weights saved for warm-start init.')

Training surrogate (5 epochs on all pairs)...
  epoch 1/5  loss=0.6364
  epoch 2/5  loss=0.5238
  epoch 3/5  loss=0.4284
  epoch 4/5  loss=0.3691
  epoch 5/5  loss=0.3374


Surrogate trained. Weights saved for warm-start init.


## Section 3 — Grey-box (`init_mode='target'`, `split_mode='member_aware'`)

The attacker has obtained the exact target authentication model checkpoint.
Each shadow LSTM is initialised from those weights. This is the attacker with maximum model access.

The shadows were trained in NB05b using `split_mode='member_aware'` with the member list
loaded from `logs/subject_split.json`. We load that checkpoint directly, no retraining.

In [11]:
with open(CKPT_GREY_SRC) as f:
    ckpt_grey = json.load(f)

out_mf_grey = {int(k): v for k, v in ckpt_grey['mf'].items()}

fpr_grey, tpr_grey, auc_grey, tpr10_grey, tpr20_grey, _, m_grey, nm_grey = \
    lira_eval(out_mf_grey, all_mf_targets, member_list, nonmember_list)

log.info(f'Grey-box (target/member-aware): AUC={auc_grey:.4f}  TPR@0.10={tpr10_grey:.3f}')
print(f'Grey-box (target/member-aware): AUC={auc_grey:.4f}  TPR@0.10={tpr10_grey:.3f}')

Grey-box (target/member-aware): AUC=0.8783  TPR@0.10=0.524


Grey-box (target/member-aware): AUC=0.8783  TPR@0.10=0.524


## Section 4 — Black-box Warm (`init_mode='warm_base'`, `split_mode='member_aware'`)

The attacker does not have the target model. They trained their own surrogate ShadowLSTM
from scratch on the available D5 pairs (Section 2) and use those weights as the shared
starting point for all K shadow LSTMs.

The subject list comes from `logs/subject_split.json`, same source as NB05b.
The gap grey-warm measures how much having the exact target weights helps
over a self-trained surrogate of the same architecture.

In [12]:
out_raw_warm, out_lf_warm, out_mf_warm = train_shadow_models(
    all_feats, y_tr, pair_subjects, member_list,
    te_feats,  y_te, te_pair_subjects, nonmember_list,
    K=K_LIRA, epochs=LIRA_EPOCHS,
    init_mode='warm_base',
    split_mode='member_aware',
    target_state=surrogate_state,
    pair_subjects_w2=subj_win2_tr,
    checkpoint_path=CKPT_WARM,
    batch_size=BATCH, device=str(DEVICE), log=log,
    dropout=DROPOUT,
)

fpr_warm, tpr_warm, auc_warm, tpr10_warm, tpr20_warm, _, m_warm, nm_warm = \
    lira_eval(out_mf_warm, all_mf_targets, member_list, nonmember_list)

log.info(f'BB warm (surrogate/member-aware): AUC={auc_warm:.4f}  TPR@0.10={tpr10_warm:.3f}')
print(f'BB warm (surrogate/member-aware): AUC={auc_warm:.4f}  TPR@0.10={tpr10_warm:.3f}')

No checkpoint found — starting from scratch.
Training 32 shadow LSTMs (init=warm_base, split=member_aware, epochs=3)...
  [ 1/32]  0.8min  ETA=24.6min
  [ 8/32]  5.6min  ETA=16.9min
  [16/32]  11.6min  ETA=11.6min
  [24/32]  16.1min  ETA=5.4min
  [32/32]  20.7min  ETA=0.0min
Done in 20.7 min
BB warm (surrogate/member-aware): AUC=0.7513  TPR@0.10=0.333


BB warm (surrogate/member-aware): AUC=0.7513  TPR@0.10=0.333


## Section 5 — Black-box Cold (`init_mode='random'`, `split_mode='member_aware'`)

The attacker knows only the model architecture, no weights, no surrogate.
Each shadow LSTM starts from independent random Xavier initialisation.

The subject list comes from `logs/subject_split.json`.
The gap warm−cold measures whether running a full surrogate training gives a
meaningful advantage over pure random initialisation.

In [13]:
out_raw_cold, out_lf_cold, out_mf_cold = train_shadow_models(
    all_feats, y_tr, pair_subjects, member_list,
    te_feats,  y_te, te_pair_subjects, nonmember_list,
    K=K_LIRA, epochs=LIRA_EPOCHS,
    init_mode='random',
    split_mode='member_aware',
    pair_subjects_w2=subj_win2_tr,
    checkpoint_path=CKPT_COLD,
    batch_size=BATCH, device=str(DEVICE), log=log,
    dropout=DROPOUT,
)

fpr_cold, tpr_cold, auc_cold, tpr10_cold, tpr20_cold, _, m_cold, nm_cold = \
    lira_eval(out_mf_cold, all_mf_targets, member_list, nonmember_list)

log.info(f'BB cold (random/member-aware): AUC={auc_cold:.4f}  TPR@0.10={tpr10_cold:.3f}')
print(f'BB cold (random/member-aware): AUC={auc_cold:.4f}  TPR@0.10={tpr10_cold:.3f}')

No checkpoint found — starting from scratch.
Training 32 shadow LSTMs (init=random, split=member_aware, epochs=3)...
  [ 1/32]  0.6min  ETA=17.6min
  [ 8/32]  4.7min  ETA=14.0min
  [16/32]  9.7min  ETA=9.7min
  [24/32]  14.9min  ETA=5.0min
  [32/32]  20.1min  ETA=0.0min
Done in 20.1 min
BB cold (random/member-aware): AUC=0.9153  TPR@0.10=0.571


BB cold (random/member-aware): AUC=0.9153  TPR@0.10=0.571


## Section 6 — Comparison

In [ ]:
nb04 = np.load(ARTIFACT_DIR / '04_mia_scores.npz')
all_sc = np.concatenate([nb04['member_scores'], nb04['nonmember_scores']])
all_lb = np.concatenate([np.ones(len(nb04['member_scores'])), np.zeros(len(nb04['nonmember_scores']))])
fpr_base, tpr_base, _ = roc_curve(all_lb, all_sc)
auc_base   = sk_auc(fpr_base, tpr_base)
tpr10_base = tpr_at_fpr(fpr_base, tpr_base, 0.10)

rows = [
    ('Simple delta (no shadows)',          auc_base,  tpr10_base,  '—'),
    ('Grey-box   target / member-aware',   auc_grey,  tpr10_grey,  'target model'),
    ('BB warm   surrogate / member-aware', auc_warm,  tpr10_warm,  'surrogate'),
    ('BB cold   random / member-aware',    auc_cold,  tpr10_cold,  'random Xavier'),
]

print(f'  {"Variant":<38}  {"AUC":>6}  {"ΔAUC":>7}  {"TPR@0.10":>9}  Init')
print('-' * 85)
for name, auc_v, t10, init in rows:
    print(f'  {name:<38}  {auc_v:.4f}  {auc_v-auc_base:>+7.4f}  {t10:.3f}       {init}')

print(f'\nAdvantage (grey  − cold): ΔAUC = {auc_grey - auc_cold:+.4f}  ← full knowledge gain')
print(f'Advantage (grey  − warm): ΔAUC = {auc_grey - auc_warm:+.4f}  ← value of exact target weights')
print(f'Advantage (warm  − cold): ΔAUC = {auc_warm - auc_cold:+.4f}  ← value of surrogate training')

log.info(f'base={auc_base:.4f}  grey={auc_grey:.4f}  warm={auc_warm:.4f}  cold={auc_cold:.4f}')

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(fpr_base, tpr_base, color='#95a5a6', linestyle='--', linewidth=1.5,
        label=f'Simple delta       (AUC={auc_base:.3f})')
ax.plot(fpr_cold, tpr_cold, color='#e74c3c', linewidth=2,
        label=f'BB cold  random    (AUC={auc_cold:.3f})')
ax.plot(fpr_warm, tpr_warm, color='#e67e22', linewidth=2,
        label=f'BB warm  surrogate (AUC={auc_warm:.3f})')
ax.plot(fpr_grey, tpr_grey, color='#2ecc71', linewidth=2,
        label=f'Grey-box target    (AUC={auc_grey:.3f})')
ax.plot([0,1],[0,1], color='lightgray', linestyle=':', linewidth=1)
ax.axvline(0.10, color='black', linestyle=':', linewidth=1, alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — Initialisation Comparison (all member-aware split)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
bins = np.linspace(0, 1, 25)
for arr, color, label in [
    (nm_grey, '#27ae60', f'NM grey   μ={nm_grey[:,1].mean():.3f}'),
    (m_grey,  '#2ecc71', f'M  grey   μ={m_grey[:,1].mean():.3f}'),
    (nm_warm, '#d35400', f'NM warm   μ={nm_warm[:,1].mean():.3f}'),
    (m_warm,  '#e67e22', f'M  warm   μ={m_warm[:,1].mean():.3f}'),
    (nm_cold, '#c0392b', f'NM cold   μ={nm_cold[:,1].mean():.3f}'),
    (m_cold,  '#e74c3c', f'M  cold   μ={m_cold[:,1].mean():.3f}'),
]:
    ax.hist(arr[:,1], bins=bins, alpha=0.45, color=color, label=label, density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2)
ax.set_xlabel('LiRA score (mean-first)')
ax.set_ylabel('Density')
ax.set_title(f'LiRA Score Distributions (K={K_LIRA})')
ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '05c_shadow_comparison.png', dpi=150)
plt.show()